# GRPO on fs_env: Qwen3.8-27B, starting from the SFT warm start

Each step draws **training-split tasks** (A3, D, E first; balanced: one of each cause in turn, a fresh random user name,
random folder names and clutter per draw), plays **a group of conversations per task** with the current LoRA (vLLM,
temperature 1), and scores **every turn** with the fs_env checker (files match the target, reply check where the turn
needs one; no LLM judge). The advantage is computed within the group for the same task and turn; groups where every
conversation got the same result teach nothing and are dropped, and a task whose group is flat twice leaves the pool. The
update is an on-policy policy-gradient step on the LoRA, then the new adapter is hot-loaded into vLLM.

**How to read it:** the reward should rise over steps; *mixed groups* is how much learning signal each step had; the
over-reaction probes must stay above the red line (they score the first move with thinking off, a coarse check: real
held-out conversations are run at the end).

**Updates:** this notebook reads JSON only. `./grpo_updater.sh start` copies the run's files from the pod every 2 min; re-run
the cells or the live cell. The run is chosen in `results/fs_env/grpo/grpo_notebook_config.json`.

In [1]:
import json, datetime, statistics
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import HTML, display
ROOT = next(p for p in (Path("."), Path("..")) if (p / "results/fs_env/grpo").exists())
CFG = json.loads((ROOT / "results/fs_env/grpo/grpo_notebook_config.json").read_text())
RUN_DIR = ROOT / CFG["local_dir"]; PRICE_H = CFG.get("price_per_hour", 1.80)
C_MAIN = "#7A4E9C"
CAT = {"A3": "#2a78d6", "D": "#eb6834", "E": "#1baf7a", "A4": "#eda100", "B": "#e87ba4", "C": "#008300", "F": "#4a3aa7", "G": "#888", "H": "#444"}
def read_json(p, d=None):
    try: return json.loads(Path(p).read_text())
    except (OSError, ValueError): return d
def read_jsonl(p):
    rows = []
    if Path(p).exists():
        for l in open(p):
            try: rows.append(json.loads(l))
            except ValueError: pass
    return rows
def fmt_min(s):
    if s is None: return "-"
    m = int(round(s / 60)); return f"{m} min" if m < 60 else f"{m // 60} h {m % 60:02d} min"
def table(rows, head=None):
    h = "".join(f"<th style='text-align:left;padding:2px 10px'>{x}</th>" for x in head) if head else ""
    b = "".join("<tr>" + "".join(f"<td style='padding:2px 10px'>{x}</td>" for x in r) + "</tr>" for r in rows)
    return f"<table style='border-collapse:collapse;font-size:13px'>{'<tr>' + h + '</tr>' if h else ''}{b}</table>"
def load():
    return (read_json(RUN_DIR / "status.json", {}), read_jsonl(RUN_DIR / "grpo_log.jsonl"), read_jsonl(RUN_DIR / "probes.jsonl"),
            {int(p.stem.split("_")[1]): read_jsonl(p) for p in sorted((RUN_DIR / "rollouts").glob("step_*.jsonl"))} if (RUN_DIR / "rollouts").exists() else {})
S, LOG, PROBES, ROLL = load()
print(f"run {CFG['run']}: stage {S.get('stage', 'not started / not fetched')}, {len(LOG)} steps logged, {len(PROBES)} probe checks, {len(ROLL)} rollout files")

run run_v1: stage finished, 30 steps logged, 7 probe checks, 30 rollout files


## 1. Run info

In [3]:
def finish_time(S):
    if S.get("eta_s") is None or not S.get("updated"): return "-"
    if str(S.get("stage", "")).startswith(("finished", "stopped")): return "done"
    t = datetime.datetime.fromisoformat(S["updated"]).replace(tzinfo=datetime.timezone.utc) + datetime.timedelta(seconds=S["eta_s"])
    return f"<b>{t.astimezone().strftime('%H:%M')}</b>"
def info_html(S, LOG, PROBES):
    a = S.get("args", {}); last = LOG[-1] if LOG else {}; pct = lambda x: f"{x:.0%}" if isinstance(x, (int, float)) else "-"
    o = [f"<h4>{CFG['run']} &middot; stage: <span style='color:{C_MAIN}'>{S.get('stage', 'not started')}</span>"
         f"{' &middot; updated ' + S['updated'].replace('T', ' ') + ' (pod clock, UTC)' if S.get('updated') else ''}</h4>"]
    o.append("<b>Progress</b>" + table([
        ["step", f"{S.get('step', 0)} / {S.get('max_steps', a.get('steps', '-'))}"],
        ["elapsed / ETA", f"{fmt_min(S.get('elapsed_s'))} / {fmt_min(S.get('eta_s'))}"],
        ["estimated finish (your time)", finish_time(S)],
        ["last step", f"reward (turns) {pct(last.get('reward_turn'))}, conversations passed {pct(last.get('reward_conv'))}, "
                      f"mixed groups {last.get('turn_groups_mixed', '-')}/{(last.get('turn_groups_mixed') or 0) + (last.get('turn_groups_flat') or 0) or '-'}, "
                      f"{last.get('train_items', '-')} trained turns, {last.get('step_s', '-')} s" if last else "-"],
        ["tasks left in the pool", last.get("pool_left", "-")],
        ["GPU memory (peak, trainer)", f"{S.get('gpu_mem_gb', '-')} GB"],
        ["GPU cost so far", f"&#36;{S['elapsed_s'] / 3600 * PRICE_H:.2f}" if S.get("elapsed_s") else "-"]]))
    o.append("<b>Settings</b>" + table([
        ["causes / tasks per step / group", f"{a.get('causes', '-')} / {a.get('tasks_per_step', '-')} / {a.get('group', '-')}"],
        ["learning rate, grad clip", f"{a.get('lr', '-')}, {a.get('max_grad_norm', '-')}"],
        ["rollouts", f"temperature {a.get('temperature', '-')}, up to {a.get('max_steps', '-')} model calls per turn"],
        ["start adapter", a.get("init_adapter") or "fresh LoRA"],
        ["probes", f"every {a.get('probe_every', '-')} steps, stop if the act rate drops more than {pct(a.get('max_act_drop'))}"]]))
    return "".join(o)
display(HTML(info_html(S, LOG, PROBES)))

step,30 / 30
elapsed / ETA,3 h 26 min / 0 min
estimated finish (your time),done
last step,"reward (turns) 80%, conversations passed 59%, mixed groups 8/12, 64 trained turns, 366 s"
tasks left in the pool,265
"GPU memory (peak, trainer)",58.9 GB
GPU cost so far,$12.98
causes / tasks per step / group,"A3,D,E / 4 / 8"
"learning rate, grad clip","5e-06, 1.0"
rollouts,"temperature 1.0, up to 10 model calls per turn"
start adapter,/root/sft/run_v1/final_adapter


In [22]:
# live view: re-reads the JSON (kept current by ./grpo_updater.sh) every 30 s until the run ends (interrupt to stop)
import time
from IPython.display import clear_output
try:
    while True:
        S, LOG, PROBES, ROLL = load(); clear_output(wait=True); display(HTML(info_html(S, LOG, PROBES)))
        if str(S.get("stage", "")).startswith(("finished", "stopped")): break
        time.sleep(30)
except KeyboardInterrupt:
    pass

step,30 / 30
elapsed / ETA,3 h 26 min / 0 min
estimated finish (your time),done
last step,"reward (turns) 80%, conversations passed 59%, mixed groups 8/12, 64 trained turns, 366 s"
tasks left in the pool,265
"GPU memory (peak, trainer)",58.9 GB
GPU cost so far,$12.98
causes / tasks per step / group,"A3,D,E / 4 / 8"
"learning rate, grad clip","5e-06, 1.0"
rollouts,"temperature 1.0, up to 10 model calls per turn"
start adapter,/root/sft/run_v1/final_adapter


---
# Graphs

In [23]:
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"; pio.templates.default = "plotly_white"
def placeholder(title, text="no data yet"):
    f = go.Figure(); f.add_annotation(text=text, showarrow=False, font=dict(size=16, color="#888"))
    f.update_layout(title=title, height=240, xaxis=dict(visible=False), yaxis=dict(visible=False)); f.show()
S, LOG, PROBES, ROLL = load()

## 2. Reward

In [24]:
if LOG:
    steps = [l["step"] for l in LOG]
    fig = go.Figure()
    fig.add_scatter(x=steps, y=[l["reward_turn"] for l in LOG], mode="lines+markers", name="turns passed (reward)", line=dict(color=C_MAIN, width=3))
    fig.add_scatter(x=steps, y=[l["reward_conv"] for l in LOG], mode="lines+markers", name="whole conversations passed", line=dict(color="#aaa", dash="dot"))
    by = defaultdict(dict)
    for st, rs in ROLL.items():
        for c in {r["cause"] for r in rs}:
            rr = [r for r in rs if r["cause"] == c]
            by[c][st] = sum(sum(t["passed"] for t in r["turns"]) / max(1, len(r["turns"])) for r in rr) / len(rr)
    for c, d in sorted(by.items()):
        xs = sorted(d); fig.add_scatter(x=xs, y=[d[x] for x in xs], mode="markers", name=f"{c} (turns passed)", marker=dict(color=CAT.get(c, "#666"), size=8, opacity=0.7))
    fig.update_layout(title="Reward per step (each step draws different tasks, so expect noise; the trend matters)", yaxis=dict(range=[0, 1.05], tickformat=".0%"),
                      xaxis_title="GRPO step", height=420, legend=dict(orientation="h", y=-0.25))
    fig.show()
else:
    placeholder("Reward per step", "no step finished yet")

## 3. Learning signal and update

In [25]:
if LOG:
    steps = [l["step"] for l in LOG]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Learning signal: (task, turn) groups per step", "Update: loss and gradient norm"), specs=[[{}, {"secondary_y": True}]])
    fig.add_bar(x=steps, y=[l["turn_groups_mixed"] for l in LOG], name="mixed (teaches)", marker_color=C_MAIN, row=1, col=1)
    fig.add_bar(x=steps, y=[l["turn_groups_flat"] for l in LOG], name="flat (dropped)", marker_color="#ccc", row=1, col=1)
    fig.add_scatter(x=steps, y=[l["loss"] for l in LOG], mode="lines+markers", name="loss", line=dict(color="#2a78d6"), row=1, col=2)
    fig.add_scatter(x=steps, y=[l["grad_norm"] for l in LOG], mode="lines", name="grad norm", line=dict(color="#eb6834", dash="dot"), row=1, col=2, secondary_y=True)
    fig.update_layout(barmode="stack", height=400, legend=dict(orientation="h", y=-0.25)); fig.show()
else:
    placeholder("Learning signal", "no step finished yet")

## 5. Every group, every step

In [26]:
if ROLL:
    steps = sorted(ROLL); width = max(len({r["group_task"] for r in ROLL[s]}) for s in steps)
    z, txt, hov = [], [], []
    for s in steps[::-1]:
        tasks = list(dict.fromkeys(r["group_task"] for r in ROLL[s]))
        zr, tr, hr = [], [], []
        for j in range(width):
            tid = tasks[j] if j < len(tasks) else None; rs = [r for r in ROLL[s] if r["group_task"] == tid]
            zr.append(sum(r["passed"] for r in rs) / len(rs) if rs else None)
            tr.append(f"<b>{sum(r['passed'] for r in rs)}/{len(rs)}</b><br>{tid.split('_')[0]} {'_'.join(tid.split('_')[2:])[:16] if tid.split('_')[1].isdigit() else 'original'}" if rs else "")
            hr.append(f"<b>{tid}</b> step {s}<br>{sum(r['passed'] for r in rs)}/{len(rs)} conversations passed" if rs else "")
        z.append(zr); txt.append(tr); hov.append(hr)
    fig = go.Figure(go.Heatmap(z=z, x=[f"task {j + 1}" for j in range(width)], y=[f"step {s}" for s in steps[::-1]], text=txt, texttemplate="%{text}", textfont=dict(size=10),
                               customdata=hov, hovertemplate="%{customdata}<extra></extra>", zmin=0, zmax=1, xgap=2, ygap=2,
                               colorscale=[[0, "#F4EFF8"], [0.5, "#B48FCF"], [1, "#4A2A66"]], colorbar=dict(title="group<br>passed", tickformat=".0%")))
    fig.update_layout(title="Every group: conversations passed out of the group (mid shades = a learning signal; white/dark = nothing to learn)",
                      height=140 + 48 * len(steps), xaxis=dict(side="top")); fig.show()
else:
    placeholder("Groups per step", "no rollouts yet")

---
# Evaluation

## 7. Held-out eval: SFT start vs GRPO checkpoints

The 73 held-out fs_env tasks (never trained on), 4 trials each, temperature 1, the same seed for every model, so all of them see the same
tasks, user names and folder names. Noise: with 44 trials at about 45% pass, ±3 passes can be chance; the other causes have 20 trials.
Models and files: `eval` in `grpo_notebook_config.json`.

In [2]:
EV = CFG.get("eval", {})
CAUSE_ORDER = ["A3", "D", "E", "A4", "B", "C", "F", "G", "H", "A1", "A2"]
M_COLORS = ["#aaa", "#C9B3DD", "#9A6FBF", C_MAIN, "#2a78d6", "#eb6834"]
def load_eval():
    hd = ROOT / EV.get("heldout_dir", "missing")
    H = {}
    for name, label in EV.get("heldout_models", []):
        rows = read_jsonl(hd / f"{name}.jsonl"); by = defaultdict(lambda: [0, 0])
        for r in rows:
            by[r["cause"]][0] += bool(r["passed"]); by[r["cause"]][1] += 1
        H[label] = {"done": len(rows), "passed": sum(bool(r["passed"]) for r in rows), "by": dict(by)}
    B = {}
    if EV.get("bfcl_model"):
        bd = ROOT / EV["bfcl_dir"]; m = EV["bfcl_model"]
        for c in EV.get("bfcl_categories", []):
            res = list(bd.glob(f"result/{m}/**/BFCL_v4_{c}_result.json")); sc = list(bd.glob(f"score/{m}/**/BFCL_v4_{c}_score.json"))
            score = None
            if sc:
                try: score = json.loads(open(sc[0]).readline())
                except ValueError: pass
            B[c] = {"done": len(read_jsonl(res[0])) if res else 0, "score": score}
    return H, B
def heldout_html(H):
    if not H: return "<i>no held-out eval configured</i>"
    per = EV.get("heldout_per_model", 0); labels = list(H); first = labels[0]
    head = ["cause"] + labels
    rows = [["<i>progress</i>"] + [f"{H[l]['done']}/{per}" + (" &#10003;" if per and H[l]['done'] >= per else " &middot; running") for l in labels]]
    for c in [c for c in CAUSE_ORDER if any(c in H[l]["by"] for l in labels)]:
        best = max((H[l]["by"].get(c, [0, 0])[0] for l in labels), default=0); row = [f"<b>{c}</b>"]
        for l in labels:
            k, n = H[l]["by"].get(c, [0, 0]); k0, n0 = H[first]["by"].get(c, [0, 0]); d = k - k0
            # a change only once this cause is complete (same number of trials as the reference); before that it's partial
            delta = ("" if l == first or not n else f" <span style='color:#888'>(running)</span>" if n < n0
                     else f" <span style='color:{'#1baf7a' if d > 0 else '#c0392b' if d < 0 else '#888'}'>({d:+d})</span>")
            row.append((f"<b>{k}/{n}</b>" if k == best and n else f"{k}/{n}") + delta if n else "-")
        rows.append(row)
    rows.append(["<b>all</b>"] + [(f"<b>{H[l]['passed']}/{H[l]['done']}</b> ({H[l]['passed'] / H[l]['done']:.1%})" if H[l]["done"] >= per
                                  else f"{H[l]['passed']}/{H[l]['done']} <span style='color:#888'>(partial: A3, D, E run first, so not comparable yet)</span>") if H[l]["done"] else "-" for l in labels])
    return "<b>Held-out fs_env: conversations passed (change vs " + first + ")</b>" + table(rows, head)
def bfcl_html(B):
    if not EV.get("bfcl_model"): return "<i>BFCL: no model chosen yet</i>"
    base = EV.get("bfcl_baseline", {}); rows = []
    for c in EV.get("bfcl_categories", []):
        b = B.get(c, {}); s = b.get("score"); bl = base.get(c, {}); tot = bl.get("total", 200)
        acc = f"<b>{s['accuracy']:.1%}</b> ({s['correct_count']}/{s['total_count']})" if s else "-"
        d = (s["accuracy"] - bl["accuracy"]) * 100 if s and bl else None
        diff = f"<span style='color:{'#1baf7a' if d > 0 else '#c0392b' if d < 0 else '#888'}'>{d:+.1f} pts</span>" if d is not None else "-"
        rows.append([c.replace("multi_turn_", ""), f"{b.get('done', 0)}/{tot}" + (" &#10003;" if s else ""), acc,
                     f"{bl['accuracy']:.1%}" if bl else "-", diff])
    return (f"<b>BFCL multi-turn: {EV['bfcl_model']}</b> vs {EV.get('bfcl_baseline_label', 'baseline')}"
            + table(rows, ["category", "generated", "accuracy", "baseline", "change"]))
H, B = load_eval()
display(HTML(heldout_html(H)))

cause,SFT v1 (start),GRPO step 10,GRPO step 20,GRPO step 30
progress,292/292 ✓,94/292 · running,94/292 · running,292/292 ✓
A3,23/44,25/44 (+2),25/44 (+2),20/44 (-3)
D,21/44,18/44 (-3),19/44 (-2),19/44 (-2)
E,14/44,4/6 (running),4/6 (running),19/44 (+5)
A4,20/20,-,-,20/20 (+0)
B,20/20,-,-,20/20 (+0)
C,20/20,-,-,20/20 (+0)
F,19/20,-,-,20/20 (+1)
G,15/20,-,-,18/20 (+3)
H,20/20,-,-,20/20 (+0)


In [ ]:
# live view of the evaluation: re-reads the result files every 30 s until everything is done (interrupt to stop)
import time
from IPython.display import clear_output
try:
    while True:
        EV = json.loads((ROOT / "results/fs_env/grpo/grpo_notebook_config.json").read_text()).get("eval", {})
        H, B = load_eval(); clear_output(wait=True)
        display(HTML(f"<i>updated {datetime.datetime.now():%H:%M:%S}</i><br>" + heldout_html(H) + "<br>" + bfcl_html(B)))
        h_done = all(v["done"] >= EV.get("heldout_per_model", 0) for v in H.values())
        b_done = bool(EV.get("bfcl_model")) and all(B.get(c, {}).get("score") for c in EV.get("bfcl_categories", []))
        if h_done and b_done: break
        time.sleep(30)
except KeyboardInterrupt:
    pass

cause,SFT v1 (start),GRPO step 10,GRPO step 20,GRPO step 30
progress,292/292 ✓,292/292 ✓,292/292 ✓,292/292 ✓
A3,23/44,25/44 (+2),25/44 (+2),20/44 (-3)
D,21/44,18/44 (-3),19/44 (-2),19/44 (-2)
E,14/44,20/44 (+6),21/44 (+7),19/44 (+5)
A4,20/20,20/20 (+0),20/20 (+0),20/20 (+0)
B,20/20,20/20 (+0),20/20 (+0),20/20 (+0)
C,20/20,20/20 (+0),20/20 (+0),20/20 (+0)
F,19/20,20/20 (+1),20/20 (+1),20/20 (+1)
G,15/20,17/20 (+2),17/20 (+2),18/20 (+3)
H,20/20,20/20 (+0),20/20 (+0),20/20 (+0)


In [ ]:
if H and any(v["done"] for v in H.values()):
    causes = [c for c in CAUSE_ORDER if any(c in v["by"] for v in H.values())]
    fig = go.Figure()
    for i, (label, v) in enumerate(H.items()):
        ys = [v["by"][c][0] / v["by"][c][1] if c in v["by"] and v["by"][c][1] else None for c in causes]
        tx = [f"{v['by'][c][0]}/{v['by'][c][1]}" if c in v["by"] else "" for c in causes]
        fig.add_bar(x=causes, y=ys, name=f"{label} ({v['done']} done)", marker_color=M_COLORS[i % len(M_COLORS)], text=tx, textposition="outside", textfont=dict(size=9))
    fig.update_layout(title="Held-out pass rate per cause (A3, D, E = the trained causes)", barmode="group", yaxis=dict(range=[0, 1.15], tickformat=".0%"),
                      height=440, legend=dict(orientation="h", y=-0.15))
    fig.show()
else:
    placeholder("Held-out pass rate per cause", "no held-out results yet")

## 8. BFCL multi-turn on the chosen checkpoint

Same settings as the base-model baseline (temperature 0.001, thinking on, 16,384 tokens per call), except the context limit: 131k instead of 65k
(the baseline lost 11 long_context entries to the limit). Order: miss_func and miss_param first, then base and long_context.
Model and baseline: `eval.bfcl_*` in the config.

In [ ]:
H, B = load_eval()
display(HTML(bfcl_html(B)))
if B and any(v.get("score") for v in B.values()):
    cats = [c for c in EV["bfcl_categories"] if B.get(c, {}).get("score")]
    fig = go.Figure()
    fig.add_bar(x=[c.replace("multi_turn_", "") for c in cats], y=[EV["bfcl_baseline"][c]["accuracy"] for c in cats], name=EV.get("bfcl_baseline_label", "baseline"), marker_color="#aaa")
    fig.add_bar(x=[c.replace("multi_turn_", "") for c in cats], y=[B[c]["score"]["accuracy"] for c in cats], name=EV["bfcl_model"], marker_color=C_MAIN)
    fig.update_layout(title="BFCL multi-turn accuracy", barmode="group", yaxis=dict(range=[0, 1], tickformat=".0%"), height=380, legend=dict(orientation="h", y=-0.2))
    fig.show()